In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn

# load data once — all models will use this
df_deaths = pd.read_csv('../data/Deaths_1x1.txt', skiprows=2, sep=r'\s+')
df_deaths['Age'] = df_deaths['Age'].replace('110+', 110).astype(int)
df_exposure = pd.read_csv('../data/Exposures_1x1.txt', skiprows=2, sep=r'\s+')
df_exposure['Age'] = df_exposure['Age'].replace('110+', 110).astype(int)

df = df_deaths.merge(df_exposure, on=['Year', 'Age'], suffixes=('_deaths', '_exp'))
df['mx_total'] = df['Total_deaths'] / df['Total_exp']
df = df[(df['Age'] <= 100) & (df['mx_total'] > 0)].copy()
df['log_mx_total'] = np.log(df['mx_total'])
df['log_age'] = np.log1p(df['Age'])
df['age_squared'] = df['Age'] ** 2

# standard train/test split used across all models
train_mask = df['Year'] <= 2000
test_mask = df['Year'] > 2000

print(f"Train: {train_mask.sum()} rows, Test: {test_mask.sum()} rows")
print(f"Test years: {df[test_mask]['Year'].min()} to {df[test_mask]['Year'].max()}")

Train: 8080 rows, Test: 2121 rows
Test years: 2001 to 2021


In [3]:
# ── MODEL 1: LEE-CARTER ──────────────────────────────────────────────────────
matrix = df.pivot(index='Age', columns='Year', values='mx_total')
log_mx = np.log(matrix.values)
ages = matrix.index.values
years = matrix.columns.values

# fit on training data only
matrix_train = matrix[matrix.columns[matrix.columns <= 2000]]
log_mx_train = np.log(matrix_train.values)
years_train = matrix_train.columns.values

alpha_lc = log_mx_train.mean(axis=1)
centred = log_mx_train - alpha_lc[:, np.newaxis]
U, S, Vt = np.linalg.svd(centred)
beta_lc = U[:, 0] / U[:, 0].sum()
kappa_lc = Vt[0, :] * S[0] * U[:, 0].sum()

post1950 = years_train >= 1950
lr_lc = LinearRegression()
lr_lc.fit(years_train[post1950].reshape(-1, 1), kappa_lc[post1950])

test_years = years[years > 2000]
kappa_fore = lr_lc.predict(test_years.reshape(-1, 1))
pred_log_lc = alpha_lc[:, np.newaxis] + beta_lc[:, np.newaxis] * kappa_fore[np.newaxis, :]

# get actual test values
matrix_test = matrix[matrix.columns[matrix.columns > 2000]]
actual_log_test = np.log(matrix_test.values)

mae_lc = mean_absolute_error(actual_log_test.flatten(), pred_log_lc.flatten())
rmse_lc = np.sqrt(mean_squared_error(actual_log_test.flatten(), pred_log_lc.flatten()))
print(f"Lee-Carter  — MAE: {mae_lc:.4f}  RMSE: {rmse_lc:.4f}")

Lee-Carter  — MAE: 0.2367  RMSE: 0.2896


In [4]:
# ── MODEL 2: XGBOOST ─────────────────────────────────────────────────────────
features = ['Age', 'Year', 'log_age', 'age_squared']
X = df[features].values
y = df['log_mx_total'].values

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

model_xgb = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.8, random_state=42, verbosity=0
)
model_xgb.fit(X_train, y_train, verbose=False)
pred_xgb = model_xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
print(f"XGBoost     — MAE: {mae_xgb:.4f}  RMSE: {rmse_xgb:.4f}")

XGBoost     — MAE: 0.2671  RMSE: 0.3370


In [ ]:
# ── MODEL 3: NEURAL NETWORK ──────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test_scaled)

from torch.utils.data import DataLoader, TensorDataset

class MortalityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.network(x).squeeze()

model_nn = MortalityNet()
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)
criterion = nn.MSELoss()
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

for epoch in range(1000):
    model_nn.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        criterion(model_nn(X_batch), y_batch).backward()
        optimizer.step()

model_nn.eval()
with torch.no_grad():
    pred_nn = model_nn(X_test_t).numpy()

mae_nn = mean_absolute_error(y_test, pred_nn)
rmse_nn = np.sqrt(mean_squared_error(y_test, pred_nn))
print(f"Neural Net  — MAE: {mae_nn:.4f}  RMSE: {rmse_nn:.4f}")

Neural Net  — MAE: 0.1769  RMSE: 0.2679
